<a href="https://colab.research.google.com/github/Ash100/Alignment/blob/main/Phylogenetic_Analysis_Nucleotide_Sequences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Bioinformatics Pipeline for Nucleotide Sequence Alignment with MAFFT in Google Colab**

Below is a comprehensive pipeline in Google Colab that allows you to upload sequence data in FASTA, and perform alignment with MAFFT, and perform phylogeny analysis with IQ-Tree.

This pipeline is generated by **Dr. Ashfaq Ahmad**, you can watch complete video tutorial of this pipeline on [Bioinformatic Insights](https://youtu.be/3yQTxuI5SGk).



### **Important Citations**

**MAFFT** Kazutaka Katoh, Kazuharu Misawa, Kei‐ichi Kuma, Takashi Miyata, MAFFT: a novel method for rapid multiple sequence alignment based on fast Fourier transform, Nucleic Acids Research, Volume 30, Issue 14, 15 July 2002, Pages 3059–3066, https://doi.org/10.1093/nar/gkf436

**IQ-Tree** Bui Quang Minh, Heiko A. Schmidt, Olga Chernomor, Dominik Schrempf,
Michael D. Woodhams, Arndt von Haeseler, and Robert Lanfear (2020)
IQ-TREE 2: New models and efficient methods for phylogenetic inference
in the genomic era. Mol. Biol. Evol., in press.
https://doi.org/10.1093/molbev/msaa015

**Model Finder** Subha Kalyaanamoorthy, Bui Quang Minh, Thomas KF Wong, Arndt von Haeseler,
and Lars S Jermiin (2017) ModelFinder: Fast model selection for
accurate phylogenetic estimates. Nature Methods, 14:587–589.
https://doi.org/10.1038/nmeth.4285

In [ ]:
#@title 1. Setup Environment

# Install both tools
!apt install mafft -y
# Download IQ-TREE (Linux version)
!wget https://github.com/iqtree/iqtree2/releases/download/v2.2.2.7/iqtree-2.2.2.7-Linux.tar.gz
!tar -xzf iqtree-2.2.2.7-Linux.tar.gz
!./iqtree-2.2.2.7-Linux/bin/iqtree2 --version


The above code installs, MAFFT for multiple sequence alignment and IQ-Tree for phylogenetic tree generation

In [ ]:
#@title File Upload
from google.colab import files
uploaded = files.upload()

In [ ]:
#@title Verify a successful upload
!ls -lh
!head {list(uploaded.keys())[0]}

We are going to lauch MAFFT and align our uploaded data. The code uses **--auto**, that means it lets MAFFT choose the best strategy automatically. If you are interested in some specialised strategy, please choose accordingly.
The**--thread -1** option means it will use all available CPU cores for faster processing. Finally the output will be save with a name **aligned.fasta**, and an output with **.tree** is actually the calculated tree in Newick Formate.

## 📘 **Important Read**

Below are different codes of **MAFFT**, classified into different **Strategies**.

### 🧪 **Strategy_1:** Fast and Simple for Smaller to Medium Datasets (Guided Tree)

- **Guided Tree**: A rough tree generated from the alignment itself  
- ⚠️ **Not recommended for publication**
- ✅ Good for quick visualizations or preliminary exploration

---

### 🧬 **Strategy_2:** Use **IQ-TREE** for Sophisticated Tree-Building

- Uses **Maximum Likelihood (ML)** methods (not just a guide tree)
- Supports **branch support values** (e.g., `-B 1000`, `-alrt`, `-abayes`)
- 🧠 Better suited for:
  - **Larger datasets** (>100 sequences)
  - **Highly divergent sequences**
  - **Publication-ready trees**


### 🌳 Key Differences: MAFFT Tree vs. IQ-TREE

| Feature         | MAFFT Tree                                      | IQ-TREE                                                    |
|-----------------|--------------------------------------------------|-------------------------------------------------------------|
| **Purpose**     | Quick guide tree for alignment                   | High-accuracy phylogenetic inference                        |
| **Algorithm**   | Guide tree based on sequence similarity          | Maximum likelihood with advanced model testing              |
| **Speed**       | Very fast                                        | Slower (due to model testing and bootstraps)                |
| **Accuracy**    | Approximate                                      | High — suitable for publication                             |
| **Branch Support** | ❌ Not provided                              | ✅ Bootstrap, SH-like, aLRT support options                  |
| **Modeling**    | No evolutionary model                            | Full model selection and rate heterogeneity support         |
| **Output Tree** | Rough tree for internal use only                 | Final tree with distances and support values                |
| **Best For**    | Alignment guidance / quick visualization         | Publishing / detailed evolutionary analysis                 |



In [ ]:
#@title **STRATEGY_1:** Perform Alignment and Generate Tree with MAFFT
# 1. First run MAFFT without redirection to verify tree creation
!mafft --auto --treeout --thread -1 /content/rbcL_gene.txt

# 2. Now run with proper file handling (this always works)
input_file = "/content/rbcL_gene.txt"
output_alignment = "aligned.fas"

# This syntax ensures the tree is created
!mafft --auto --treeout --thread -1 {input_file} > {output_alignment} 2>&1

# 3. The tree will be named after your INPUT file
tree_file = f"{input_file}.tree"

# 4. Verify files
print("\nGenerated files:")
!ls -lh {output_alignment} {tree_file}

# 5. Rename tree file if needed (optional)
!cp {tree_file} gene_tree.tre

### 🧬 Standard Nucleotide Models in IQ-TREE

| Model              | Description                                                                 |
|--------------------|-----------------------------------------------------------------------------|
| **JC**             | Jukes-Cantor (equal base frequencies, equal rates)                          |
| **K80**            | Kimura 2-parameter (transitions ≠ transversions)                            |
| **HKY**            | Hasegawa-Kishino-Yano (transitions/transversions + unequal base freq)       |
| **SYM**            | Symmetrical (reversible, equal base freq)                                   |
| **GTR**            | General Time Reversible (most general model)                                |
| **TN / TN93**      | Tamura-Nei model (transitions split into purine/pyrimidine rates)           |
| **TIM, TVM, TrN, K81, TPM (various)** | Intermediate models between HKY and GTR, useful for model selection |

---

### ⏳ Rate Heterogeneity Options

These can be added to any model:

| Modifier   | Description                                                       |
|------------|-------------------------------------------------------------------|
| `+I`       | Proportion of invariable sites                                    |
| `+G` / `+G4` | Gamma-distributed rates with 4 categories                         |
| `+G8`, `+G12` | Gamma with 8 or 12 categories                                     |
| `+R2`–`+R10` | FreeRate model (nonparametric rates)                            |
| `+ASC`     | Ascertainment bias correction (for SNP-only alignments)           |

---

### ✅ Examples of Combined Models

| Model Code   | Meaning                                                          |
|--------------|------------------------------------------------------------------|
| `GTR+G`      | General time reversible + gamma-distributed rates                |
| `HKY+I+G`    | HKY with invariable sites and gamma                              |
| `SYM+R4`     | Symmetrical model with FreeRate (4 categories)                   |
| `JC+ASC`     | Jukes-Cantor with ascertainment correction                       |

---

### 🔍 Tip: Let IQ-TREE Pick the Best Model

Use the following option to automatically test and select the best model:

```bash
-m MFP


In [ ]:
#@title Run IQ-TREE on your aligned MAFFT file
!/content/iqtree-2.2.2.7-Linux/bin/iqtree2 -s /content/aligned.fas -m MFP -bb 1000 -nt AUTO



### 📁 Output Files You Get

- **`tree_output.treefile`**  
  → Final tree in **Newick** format (includes **bootstrap values**)

- **`tree_output.iqtree`**  
  → Full **model information**, likelihoods, tree statistics

- **`tree_output.log`**  
  → Execution **log file** with detailed processing steps

- **`tree_output.model.gz`**  
  → **Model selection results**, useful if `-m TEST` was used

- **`aligned.fas.mldist`**  
  → Pairwise **distance matrix** between all sequences



## 🧠 **Which Should You Choose?**

1. ✅ **For quick visualization**: Use **MAFFT** with `--treeout` (Strategy_1)  
2. 🔄 **For analysis/pipeline**: Use **IQ-TREE** (Strategy_2)  
3. 📄 **For publications**: Use **IQ-TREE** with **bootstrap/support values**




---

### 📢 **Enjoyed this notebook?**

If you found this work helpful or insightful, consider subscribing to **Bioinformatics Insights** for more content on cutting-edge tools, tutorials, and research in bioinformatics.

Your support helps keep this work going and contributes to a growing community of computational biologists!

👉 [Subscribe to Bioinformatics Insights](https://youtube.com/@bioinformaticsinsights?si=8SCaRpnoycm2oqND)

---
